# TRIBE v2 — Colab Backend
**Run Cell 1 → Restart Runtime → Run Cells 2–7**

In [ ]:
# CELL 1 — Fix numpy first, then RESTART RUNTIME before continuing
!pip install -q 'numba>=0.61.0'           # supports numpy 2.x
!pip install -q 'numpy==2.2.6' --force-reinstall
import numpy as np
print(f'numpy: {np.__version__}')
print()
print('>>> RESTART RUNTIME NOW, then run from Cell 2 <<<')
print('    Colab : Runtime → Restart runtime')
print('    VSCode: Kernel toolbar → Restart')

In [ ]:
# CELL 2 — Install everything (after restart)
import numpy as np, sys, subprocess, os
assert np.__version__ == '2.2.6', f'Wrong numpy {np.__version__} — did you restart after Cell 1?'
print(f'numpy {np.__version__} ✓')

# ffmpeg
!apt-get install -y -q ffmpeg 2>/dev/null || echo 'ffmpeg skipped'

# torch — skip if already present
r = subprocess.run([sys.executable, '-c', 'import torch; print(torch.__version__)'],
                   capture_output=True, text=True)
if r.returncode == 0:
    print(f'torch {r.stdout.strip()} ✓')
else:
    !pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

# tribev2 — install from GitHub (proper site-packages registration, not editable)
!pip install -q --no-deps 'tribev2 @ git+https://github.com/facebookresearch/tribev2.git'
# tribev2's deps (skip numpy/torch — already correct versions)
!pip install -q einops omegaconf 'x-transformers' huggingface_hub \
    moviepy soundfile gtts spacy langdetect Levenshtein transformers accelerate

# whisperx (after torch)
!pip install -q whisperx

# backend deps
!pip install -q fastapi 'uvicorn[standard]' yt-dlp pydantic nest_asyncio

# re-pin numpy last — whisperx/other packages try to downgrade it
!pip install -q 'numpy==2.2.6' --force-reinstall

# verify
import importlib, numpy as np, torch
importlib.invalidate_caches()
import tribev2
from tribev2 import TribeModel
print(f'\nnumpy   : {np.__version__}')
print(f'torch   : {torch.__version__}')
print(f'tribev2 : {tribev2.__file__}')
print(f'TribeModel ✓')

In [ ]:
# CELL 3 — Clone / update brain-neuro
import os
repo = '/content/brain-neuro'
if os.path.exists(repo):
    !git -C {repo} pull -q
    print('brain-neuro updated ✓')
else:
    !git clone -q https://github.com/Sambhram1/brain-neuro.git {repo}
    print('brain-neuro cloned ✓')
os.chdir(repo)
print(f'cwd: {os.getcwd()}')

In [ ]:
# CELL 4 — HuggingFace login (weights download automatically on first request)
import huggingface_hub
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
    print('HF_TOKEN from Colab secrets ✓')
except Exception:
    hf_token = input('Enter HuggingFace token: ')
huggingface_hub.login(token=hf_token, add_to_git_credential=False)
print('Logged in ✓ — weights will download on first /analyze call')

In [ ]:
# CELL 5 (Optional) — cookies.txt for Instagram / TikTok  (skip for YouTube)
import shutil, os
try:
    from google.colab import files
    print('Upload cookies.txt or skip this cell')
    uploaded = files.upload()
    for fname in uploaded:
        shutil.move(fname, 'cookies.txt')
        print('cookies.txt saved ✓')
except ImportError:
    print('cookies.txt found ✓' if os.path.exists('cookies.txt') else 'No cookies.txt — YouTube only')

In [ ]:
# CELL 6 — Install cloudflared
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 \
    -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
print('cloudflared ✓')

In [ ]:
# CELL 7 — Start backend + tunnel
import nest_asyncio, uvicorn, threading, subprocess, time, re, os
os.chdir('/content/brain-neuro')
nest_asyncio.apply()

def run_server():
    uvicorn.run('backend:app', host='0.0.0.0', port=8000, log_level='warning')

threading.Thread(target=run_server, daemon=True).start()
time.sleep(4)
print('FastAPI running on :8000 ✓')

proc = subprocess.Popen(
    ['/usr/local/bin/cloudflared', 'tunnel', '--url', 'http://localhost:8000'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT
)
print('Waiting for tunnel...')
for line in proc.stdout:
    match = re.search(r'https://[\w-]+\.trycloudflare\.com', line.decode())
    if match:
        url = match.group()
        print(f'\n{"═"*50}')
        print(f'  BACKEND URL: {url}')
        print(f'{"═"*50}')
        print('→ Paste into the frontend Backend field')
        break